# 04 Train Specialist Model

Purpose: fine-tune RoBERTa specialist classifiers on genAI-labeled train/test data, then run the saved specialist models on the locked holdout. This notebook is safe to run in Colab or from the repo root. It does not store API keys.


## Setup

If running in Colab, mount Drive first. If running locally, leave `BASE` as the repository root. Update the label file paths only if your train/test genAI labels use different names.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DEFAULT_BASE = Path('/content/drive/MyDrive/goodreads-reader-response-classifier')
except Exception:
    DEFAULT_BASE = Path.cwd()

BASE = DEFAULT_BASE
TRANSFORMER = BASE / 'scripts' / 'train_transformer.py'
PREDICT_SPECIALIST = BASE / 'scripts' / 'predict_specialist.py'
TRAIN_TEXTS = BASE / 'data' / 'processed' / 'train_no_holdout_overlap.csv'
TEST_TEXTS = BASE / 'data' / 'processed' / 'test_no_holdout_overlap.csv'
HOLDOUT_TEXTS = BASE / 'data' / 'processed' / 'holdout_locked.csv'
TRAIN_LABELS = BASE / 'data' / 'results' / 'train_labeled.csv'
TEST_LABELS = BASE / 'data' / 'results' / 'test_labeled.csv'
print('BASE:', BASE)


## Configuration 1

RoBERTa-base, learning rate 2e-5, 3 epochs, batch size 8, max length 256, threshold 0.5.


In [ ]:
!python "$TRANSFORMER" --task emotions --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 2e-5 --epochs 3


In [ ]:
!python "$TRANSFORMER" --task commitment --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 2e-5 --epochs 3


In [ ]:
!python "$TRANSFORMER" --task recommendation --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 2e-5 --epochs 3


## Configuration 2

RoBERTa-base, learning rate 1e-5, 5 epochs, batch size 16, max length 512, threshold 0.4 for emotions.


In [ ]:
!python "$TRANSFORMER" --task emotions --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 1e-5 --epochs 5 --batch-size 16 --max-length 512 --threshold 0.4


In [ ]:
!python "$TRANSFORMER" --task commitment --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 1e-5 --epochs 5 --batch-size 16 --max-length 512


In [ ]:
!python "$TRANSFORMER" --task recommendation --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 1e-5 --epochs 5 --batch-size 16 --max-length 512


## Configuration 3, Selected Specialist Model

RoBERTa-base, learning rate 2e-5, 5 epochs, batch size 16, max length 512, threshold 0.4 for emotions.


In [ ]:
!python "$TRANSFORMER" --task emotions --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 2e-5 --epochs 5 --batch-size 16 --max-length 512 --threshold 0.4


In [ ]:
!python "$TRANSFORMER" --task commitment --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 2e-5 --epochs 5 --batch-size 16 --max-length 512


In [ ]:
!python "$TRANSFORMER" --task recommendation --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 2e-5 --epochs 5 --batch-size 16 --max-length 512


## Configuration 4

RoBERTa-base, learning rate 3e-5, 10 epochs, batch size 16, max length 512, threshold 0.4 for emotions.


In [ ]:
!python "$TRANSFORMER" --task emotions --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 3e-5 --epochs 10 --batch-size 16 --max-length 512 --threshold 0.4


In [ ]:
!python "$TRANSFORMER" --task commitment --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 3e-5 --epochs 10 --batch-size 16 --max-length 512


In [ ]:
!python "$TRANSFORMER" --task recommendation --model-name roberta-base   --train-texts "$TRAIN_TEXTS" --eval-texts "$TEST_TEXTS"   --train-labels "$TRAIN_LABELS" --eval-labels "$TEST_LABELS"   --learning-rate 3e-5 --epochs 10 --batch-size 16 --max-length 512


## Locate Saved Models

The training script saves each task model under `models/<task>_roberta-base_lr.../best_model`. Use the selected configuration 3 paths below, or change them to wherever the final model was saved in Drive.


In [ ]:
for p in (BASE / 'models').glob('**/summary.json'):
    print(p)


In [ ]:
EMOTION_MODEL = BASE / 'models' / 'emotions_roberta-base_lr2e-05_ep5.0' / 'best_model'
COMMITMENT_MODEL = BASE / 'models' / 'commitment_roberta-base_lr2e-05_ep5.0' / 'best_model'
RECOMMENDATION_MODEL = BASE / 'models' / 'recommendation_roberta-base_lr2e-05_ep5.0' / 'best_model'
HOLDOUT_PREDICTIONS = BASE / 'data' / 'results' / 'holdout_roberta_config3_predictions.csv'


## Predict Locked Holdout With Saved Specialist Models


In [ ]:
!python "$PREDICT_SPECIALIST"   --input "$HOLDOUT_TEXTS"   --output "$HOLDOUT_PREDICTIONS"   --emotion-model "$EMOTION_MODEL"   --commitment-model "$COMMITMENT_MODEL"   --recommendation-model "$RECOMMENDATION_MODEL"   --threshold 0.4 --batch-size 16 --max-length 512


## Evaluate Holdout Predictions


In [ ]:
EVALUATE = BASE / 'scripts' / 'evaluate_predictions.py'
GOLD = BASE / 'data' / 'results' / 'holdout_human_consensus.csv'
OUT = BASE / 'data' / 'results' / 'holdout_roberta_config3_eval.json'

!python "$EVALUATE" --gold "$GOLD" --pred "$HOLDOUT_PREDICTIONS" --output "$OUT"


In [ ]:
import pandas as pd

gold = pd.read_csv(GOLD)
pred = pd.read_csv(HOLDOUT_PREDICTIONS)
holdout = pd.read_csv(HOLDOUT_TEXTS)

print('gold rows:', len(gold), 'unique:', gold.review_uid.nunique())
print('pred rows:', len(pred), 'unique:', pred.review_uid.nunique())
print('holdout rows:', len(holdout), 'unique:', holdout.review_uid.nunique())
print('gold/pred overlap:', len(set(gold.review_uid) & set(pred.review_uid)))
print('holdout/pred overlap:', len(set(holdout.review_uid) & set(pred.review_uid)))
print('holdout/gold overlap:', len(set(holdout.review_uid) & set(gold.review_uid)))
print('
Prediction distributions:')
print(pred['commitment'].value_counts(dropna=False))
print(pred['recommendation'].value_counts(dropna=False))
print(pred['emotions'].value_counts(dropna=False).head(20))
